# 🗳️ Decoding the 2026 Tamil Nadu Assembly Election
**Resume Project Challenge — Codebasics**

**Data Source:** Election Commission of India (ECI) — publicly available results

**Story:** Three data stories for AtliQ Media's one-hour election show
1. The Flip Story — 70% of the map changed hands
2. The Vote Share Story — A new party rewrote the numbers
3. The Margin Story — Wins got harder to come by

---
> ⚠️ **Disclaimer:** This is a non-partisan data analysis exercise using only ECI public data. No political position is expressed or implied.

## Step 1: Install & Import Libraries

In [ ]:
# Install plotly for interactive charts (already available in Colab)
# If running locally: pip install pandas plotly

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print('✅ Libraries loaded')

## Step 2: Load Data
> Upload the 3 CSV files to Colab using the Files panel on the left, then run this cell.

In [ ]:
# Load all three datasets
df21 = pd.read_csv('tn_2021_results.csv')
df26 = pd.read_csv('tn_2026_results.csv')
master = pd.read_csv('constituency_master.csv')

print('2021 data:', df21.shape, '— rows x columns')
print('2026 data:', df26.shape)
print('Master:   ', master.shape)
print()
print('Sample — 2021:')
df21.head(3)

## Step 3: Get the Winner of Each Constituency

Each row in the data is one **candidate**. We need to find the candidate with the **most votes** in each of the 234 constituencies — that's the winner.

We also calculate:
- **win_pct** — what % of total votes did the winner get?
- **margin** — how many votes ahead of the 2nd place candidate?

In [ ]:
def get_winners(df):
    winners_list = []
    for ac in df['ac_number'].unique():
        ac_df = df[df['ac_number'] == ac]
        winner_row = ac_df.loc[ac_df['votes'].idxmax()].copy()
        total = ac_df['votes'].sum()
        sorted_ac = ac_df.sort_values('votes', ascending=False)
        second_votes = sorted_ac.iloc[1]['votes']
        winner_row['total_votes'] = total
        winner_row['win_pct'] = round(winner_row['votes'] / total * 100, 1)
        winner_row['margin'] = winner_row['votes'] - second_votes
        winners_list.append(winner_row)
    return pd.DataFrame(winners_list).reset_index(drop=True)

w21 = get_winners(df21)
w26 = get_winners(df26)

print("w21 rows:", len(w21))
print("w26 rows:", len(w26))
print(w21['party'].value_counts())

---
# 📖 Story 1: The Flip Story
**Question:** In how many constituencies did the winning party change between 2021 and 2026?

A "flip" = the party that won in 2021 is **different** from the party that won in 2026.

In [ ]:
print("w21 rows:", len(w21))
print("w26 rows:", len(w26))
print("w21 ac sample:", w21['ac_number'].tolist()[:5])
print("w26 ac sample:", w26['ac_number'].tolist()[:5])

In [ ]:
# Merge 2021 and 2026 winners on constituency number
flip_df = w21[['ac_number', 'constituency', 'party', 'region']].merge(
    w26[['ac_number', 'party']], on='ac_number', suffixes=('_21', '_26')
)

# A flip = party changed
flips = flip_df[flip_df['party_21'] != flip_df['party_26']]
held  = flip_df[flip_df['party_21'] == flip_df['party_26']]

print(f'Total constituencies : 234')
print(f'Flipped              : {len(flips)}  ({len(flips)/234*100:.0f}%)')
print(f'Same winner as 2021  : {len(held)}  ({len(held)/234*100:.0f}%)')
print()
print('Seat flows (from → to):')
flows = flips.groupby(['party_21', 'party_26']).size().reset_index(name='seats')
flows = flows.sort_values('seats', ascending=False)
print(flows.to_string(index=False))

In [ ]:
# Chart 1: Bar chart — Flipped vs Held
fig = px.bar(
    x=['Flipped (new winner)', 'Same winner as 2021'],
    y=[len(flips), len(held)],
    color=['Flipped (new winner)', 'Same winner as 2021'],
    color_discrete_map={'Flipped (new winner)': '#E63946', 'Same winner as 2021': '#457B9D'},
    text=[len(flips), len(held)],
    title='2026 Tamil Nadu: 70% of Constituencies Had a New Winner',
    labels={'x': '', 'y': 'Number of Constituencies'}
)
fig.update_traces(textposition='outside', textfont_size=16)
fig.update_layout(showlegend=False, plot_bgcolor='white', height=450)
fig.show()

In [ ]:
# Chart 2: Sankey Diagram — where did seats go?
# Only show flows with 2+ seats for clarity
flows_plot = flows[flows['seats'] >= 2].copy()

all_parties = list(set(flows_plot['party_21'].tolist() + flows_plot['party_26'].tolist()))
party_idx = {p: i for i, p in enumerate(all_parties)}

colors = ['#2196F3','#F44336','#4CAF50','#FF9800','#9C27B0','#00BCD4','#795548']
node_colors = [colors[i % len(colors)] for i in range(len(all_parties))]

fig2 = go.Figure(go.Sankey(
    node=dict(
        pad=20, thickness=25,
        label=[f"{p} (2021)" if p in flows_plot['party_21'].values else f"{p} (2026)"
               for p in all_parties],
        color=node_colors
    ),
    link=dict(
        source=[party_idx[r['party_21']] for _, r in flows_plot.iterrows()],
        target=[party_idx[r['party_26']] for _, r in flows_plot.iterrows()],
        value=flows_plot['seats'].tolist(),
        color='rgba(150,150,150,0.3)'
    )
))
fig2.update_layout(
    title='Seat Flows: Who Lost Seats to Whom? (2021 → 2026)',
    height=500
)
fig2.show()

---
# 📖 Story 2: The Vote Share Story
**Question:** Where did TVK's votes come from? How did vote share shift between 2021 and 2026?

**Vote share** = what % of all votes (excluding NOTA) did a party get across all 234 constituencies.

In [ ]:
def state_vote_share(df):
    no_nota = df[df['party'] != 'NOTA']
    total = no_nota['votes'].sum()
    share = (no_nota.groupby('party')['votes'].sum() / total * 100).round(1)
    return share.sort_values(ascending=False)

vs21 = state_vote_share(df21)
vs26 = state_vote_share(df26)

print('2021 State-wide Vote Share (top 8):')
print(vs21.head(8).to_string())
print()
print('2026 State-wide Vote Share (top 8):')
print(vs26.head(8).to_string())

In [ ]:
# Key headline numbers
dmk_drop = vs21['DMK'] - vs26['DMK']
aiadmk_drop = vs21['AIADMK'] - vs26['AIADMK']
tvk_share = vs26['TVK']

print(f"DMK:    {vs21['DMK']}% (2021) → {vs26['DMK']}% (2026)  |  DROP of {dmk_drop:.1f} pts")
print(f"AIADMK: {vs21['AIADMK']}% (2021) → {vs26['AIADMK']}% (2026)  |  DROP of {aiadmk_drop:.1f} pts")
print(f"TVK:    new party — debuted at {tvk_share}% in 2026")
print(f"\nDMK + AIADMK combined dropped: {dmk_drop + aiadmk_drop:.1f} pts")
print(f"TVK absorbed: {tvk_share:.1f} pts")

In [ ]:
# Chart 3: Vote share comparison 2021 vs 2026
parties = ['TVK', 'DMK', 'AIADMK', 'INC', 'NTK', 'PMK', 'BJP']
share_2021 = [0, vs21.get('DMK',0), vs21.get('AIADMK',0), vs21.get('INC',0),
              vs21.get('NTK',0), vs21.get('PMK',0), vs21.get('BJP',0)]
share_2026 = [vs26.get('TVK',0), vs26.get('DMK',0), vs26.get('AIADMK',0),
              vs26.get('INC',0), vs26.get('NTK',0), vs26.get('PMK',0), vs26.get('BJP',0)]

fig3 = go.Figure()
fig3.add_trace(go.Bar(name='2021', x=parties, y=share_2021,
                      marker_color='#457B9D', text=[f'{v}%' if v>0 else 'N/A' for v in share_2021],
                      textposition='outside'))
fig3.add_trace(go.Bar(name='2026', x=parties, y=share_2026,
                      marker_color='#E63946', text=[f'{v}%' for v in share_2026],
                      textposition='outside'))
fig3.update_layout(
    barmode='group',
    title='Vote Share Shift: DMK Lost 13.7 pts — TVK Debuted at 35.1%',
    yaxis_title='Vote Share (%)',
    plot_bgcolor='white',
    height=500
)
fig3.show()

In [ ]:
# Chart 4: TVK vs DMK vote share by region
regions = ['Chennai Metro', 'Kongu', 'South', 'North', 'Delta', 'Central']
tvk_by_region, dmk21_by_region, dmk26_by_region = [], [], []

for region in regions:
    d21 = df21[df21['region'] == region]
    d26 = df26[df26['region'] == region]
    t21 = d21[d21['party']!='NOTA']['votes'].sum()
    t26 = d26[d26['party']!='NOTA']['votes'].sum()
    tvk_by_region.append(round(d26[d26['party']=='TVK']['votes'].sum()/t26*100, 1))
    dmk21_by_region.append(round(d21[d21['party']=='DMK']['votes'].sum()/t21*100, 1))
    dmk26_by_region.append(round(d26[d26['party']=='DMK']['votes'].sum()/t26*100, 1))

fig4 = go.Figure()
fig4.add_trace(go.Bar(name='DMK 2021', x=regions, y=dmk21_by_region,
                      marker_color='#457B9D'))
fig4.add_trace(go.Bar(name='DMK 2026', x=regions, y=dmk26_by_region,
                      marker_color='#A8DADC'))
fig4.add_trace(go.Bar(name='TVK 2026', x=regions, y=tvk_by_region,
                      marker_color='#E63946'))
fig4.update_layout(
    barmode='group',
    title='TVK Matched or Exceeded DMK\'s 2021 Share in Every Region',
    yaxis_title='Vote Share (%)',
    plot_bgcolor='white',
    height=500
)
fig4.show()

---
# 📖 Story 3: The Margin of Victory Story
**Question:** How did the average margin of victory change? Did winners dominate or scrape through?

**Margin** = winner's votes minus 2nd place votes. A smaller margin = closer race.

In [ ]:
print('=== Margin of Victory: 2021 vs 2026 ===')
print(f"Average margin  2021: {w21['margin'].mean():,.0f} votes")
print(f"Average margin  2026: {w26['margin'].mean():,.0f} votes")
drop = w21['margin'].mean() - w26['margin'].mean()
print(f"Drop in margin      : {drop:,.0f} votes  ({drop/w21['margin'].mean()*100:.0f}% smaller)")
print()
print(f"Won with >50% of votes  2021: {(w21['win_pct']>50).sum()} seats")
print(f"Won with >50% of votes  2026: {(w26['win_pct']>50).sum()} seats")
print()
print(f"Won with <35% of votes  2021: {(w21['win_pct']<35).sum()} seats")
print(f"Won with <35% of votes  2026: {(w26['win_pct']<35).sum()} seats")
print()
print(f"Median win%  2021: {w21['win_pct'].median():.1f}%")
print(f"Median win%  2026: {w26['win_pct'].median():.1f}%")

In [ ]:
# Chart 5: Distribution of win % in 2021 vs 2026
fig5 = go.Figure()
fig5.add_trace(go.Histogram(
    x=w21['win_pct'], name='2021',
    marker_color='#457B9D', opacity=0.7,
    xbins=dict(start=20, end=75, size=3)
))
fig5.add_trace(go.Histogram(
    x=w26['win_pct'], name='2026',
    marker_color='#E63946', opacity=0.7,
    xbins=dict(start=20, end=75, size=3)
))
fig5.update_layout(
    barmode='overlay',
    title='Winners\' Vote Share Shifted Left in 2026 — Races Got Closer',
    xaxis_title='Winner\'s % of Total Votes',
    yaxis_title='Number of Constituencies',
    plot_bgcolor='white',
    height=450
)
fig5.show()

In [ ]:
# Chart 6: Avg margin by party in 2026
margin_by_party = w26.groupby('party')['margin'].mean().round(0).sort_values(ascending=False)
margin_by_party = margin_by_party[margin_by_party > 0]  # Only parties that won seats

fig6 = px.bar(
    x=margin_by_party.index,
    y=margin_by_party.values,
    text=[f'{int(v):,}' for v in margin_by_party.values],
    title='Average Winning Margin by Party — 2026',
    labels={'x': 'Party', 'y': 'Avg Margin (votes)'},
    color=margin_by_party.values,
    color_continuous_scale='RdYlGn'
)
fig6.update_traces(textposition='outside')
fig6.update_layout(plot_bgcolor='white', height=450, showlegend=False)
fig6.show()

---
# 📋 Summary: The 3 Headlines for AtliQ Media

| # | Story | Headline |
|---|-------|----------|
| 1 | **The Flip Story** | 163 of 234 constituencies had a new winner in 2026 — 70% of the map changed hands |
| 2 | **The Vote Share Story** | A brand-new party (TVK) debuted at 35.1% — more than DMK (24.3%) or AIADMK (21.3%) |
| 3 | **The Margin Story** | Average winning margin fell 43% — 2026 was won in close fights, not walkovers |

---
# ⚠️ Data Limitations

1. **2026 turnout column is blank** — ECI Form-20 (final audited data) had not been released when this dataset was prepared. Turnout-based analysis was excluded.
2. **Alliances not modelled** — parties contested as alliances; seat-level analysis treats each party independently.
3. **No demographic data used** — census data was intentionally excluded to keep analysis non-partisan.
4. **2026 data is pre-Form-20** — minor vote count revisions may occur in the final ECI audit.